# Player Identification Baseline

This reference notebook loads official CSV game records, extracts features, trains an embedding model and generates `submission_player.csv`. The official release defines the authoritative data and submission format. The architecture, features and training approach may be adapted.

## Task
Given 5–20 games from an unknown player, identify that player among 400 candidates and return five ordered candidates.

## Environment requirements
| Item | Reference requirement |
|---|---|
| GPU | At least 4 GB VRAM; approximately 2 GB training and 4 GB inference |
| Disk | Approximately 15 GB features plus a 16 MB index |
| Packages | torch, sgfmill, pandas, numpy, matplotlib |

A Linux GPU environment is recommended. Extraction produces approximately one million small files, so available disk capacity and filesystem file limits should be checked. Reduce `GPU_BATCH` for inference or `BATCH_SIZE` for training if memory is insufficient. The original tutorial estimates about 1 GB inference VRAM at `GPU_BATCH = 1024`.

The notebook uses Linux `fork` multiprocessing with notebook-defined workers. On Windows/macOS, the original guidance suggests `NPROC = 1` or moving workers into an importable `.py` module. Platform compatibility still requires testing.

The paths in the configuration cell must match the dataset location.


In [ ]:
import glob
import gzip
import json
import os
import time
from multiprocessing import Pool, Process

import numpy as np
import pandas as pd
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, IterableDataset

from utils import SGFParsePlayerIdentification
from network import GoPlayerResNet

print('torch', torch.__version__, '| cuda', torch.version.cuda)

In [ ]:
# Official data locations: preserve the released training/ and tests/ structure
TRAIN_DIR = './training'
TEST_DIR = './tests'
FEAT_DIR = './player-features'
INDEX_PATH = './player_index.json'
CKPT_DIR = './player-trained-models'

NPROC = 8              # Worker processes for feature extraction and SGF parsing
MIN_GAMES = 2          # Triplets require two different games from the same player
SEED = 42

BATCH_SIZE = 100
LEARNING_RATE = 1e-4
MARGIN = 0.5
NUM_EPOCHS = 5
STEPS_PER_EPOCH = 15_000
LOG_INTERVAL = 5_000
TOP_N = 5              # Submission requires Top-5

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

os.makedirs(CKPT_DIR, exist_ok=True)
print('device', DEVICE)
print('Total training: %s steps = %s triplets'
      % (format(NUM_EPOCHS * STEPS_PER_EPOCH, ','),
         format(NUM_EPOCHS * STEPS_PER_EPOCH * BATCH_SIZE, ',')))

The shared training pool contains ten rank CSV files, each with 100,000 games, for a total of one million games. Columns are `player_id, game_id, rank, color, sgf_content`.

Player identification uses `player_id` to identify positive pairs, `color` for the target player's side and `sgf_content` for game records. Players with fewer than two games are removed because anchor and positive examples must come from different games of the same player.


In [ ]:
df = pd.concat([pd.read_csv(p) for p in sorted(glob.glob(f'{TRAIN_DIR}/train_*.csv'))],
               ignore_index=True)

counts = df['player_id'].value_counts()
keep = set(counts[counts >= MIN_GAMES].index)
dropped = len(counts) - len(keep)
df = df[df['player_id'].isin(keep)].reset_index(drop=True)

print('Usable: %s games / %s players (minimum %d games; %d players removed)'
      % (format(len(df), ','), format(len(keep), ','), MIN_GAMES, dropped))
print('Games per player: median %d, maximum %d' % (counts[list(keep)].median(),
                                            counts[list(keep)].max()))
df.head(3)

# 1. Feature Extraction

This is a one-time preprocessing stage. Later training runs can start at Section 2 after imports and configuration.

`SGFParsePlayerIdentification` replays each game and creates history windows with shape `(17, 19, 19)`:
- Eight own-stone history planes.
- Eight opponent-stone history planes.
- One target-color plane.

The resulting windows represent local playing patterns that the embedding model learns to compare.


In [ ]:
_parser = SGFParsePlayerIdentification()
_row = df.iloc[0]
_windows = _parser.extract_features(_row['sgf_content'],
                                    str(_row['color']).lower(), mode='inference')
print('This game produced %d windows, each with shape %s'
      % (len(_windows), np.asarray(_windows[0]).shape))

### Saving one compressed file per game

Features are stored in one gzip-compressed NumPy file per game rather than kept entirely in memory. Binary masks compress well: the original tutorial reports roughly 863 KB uncompressed versus 20 KB stored per game. Sampling one window requires decompressing its game file.

`player_index.json` maps players to their game files for triplet sampling.

Filenames include `{rank}_{game_id}` because game identifiers are unique only within each rank CSV. The prefix prevents overwriting games from other rank files.


In [ ]:
def extract_worker(df_part, wid, shard_path):
    """Extract features for one partition; each worker owns a parser and index shard."""
    parser = SGFParsePlayerIdentification()
    written = bad = 0
    index = {}
    t0 = time.time()

    for _, row in df_part.iterrows():
        key = '%s_%s' % (row['rank'], row['game_id'])
        try:
            windows = parser.extract_features(
                row['sgf_content'], str(row['color']).lower(), mode='inference')
        except Exception:
            bad += 1
            continue
        if not len(windows):
            bad += 1
            continue

        arr = np.asarray(windows, dtype=np.uint8)
        with gzip.open('%s/%s.npy.gz' % (FEAT_DIR, key), 'wb', compresslevel=6) as f:
            np.save(f, arr)
        index.setdefault(row['player_id'], []).append(key)
        written += 1
        if written % 20000 == 0:
            print('  [w%d] %d/%d  %.0f min'
                  % (wid, written, len(df_part), (time.time() - t0) / 60), flush=True)

    with open(shard_path, 'w') as f:
        json.dump(index, f)
    print('  [w%d] Wrote %d games (%d parse failures), %.1f min'
          % (wid, written, bad, (time.time() - t0) / 60), flush=True)


def extract_all(df):
    """Extract in parallel, merge indexes and verify nonempty output."""
    os.makedirs(FEAT_DIR, exist_ok=True)

    procs, shards = [], []
    for i in range(NPROC):
        shard = '%s/.index_w%d.json' % (FEAT_DIR, i)
        shards.append(shard)
        p = Process(target=extract_worker, args=(df.iloc[i::NPROC], i, shard))
        p.start()
        procs.append(p)
    for p in procs:
        p.join()
        if p.exitcode != 0:
            raise SystemExit('Worker exited abnormally (exitcode %s)' % p.exitcode)

    merged = {}
    for shard in shards:
        with open(shard) as f:
            for pid, keys in json.load(f).items():
                merged.setdefault(pid, []).extend(keys)
        os.remove(shard)

    usable = {pid: ks for pid, ks in merged.items() if len(ks) >= MIN_GAMES}
    with open(INDEX_PATH, 'w') as f:
        json.dump(usable, f)

    files = glob.glob(FEAT_DIR + '/*.npy.gz')
    total = sum(os.path.getsize(x) for x in files)
    avg = total / len(files) if files else 0
    print('Created %s files, %.2f GB, average %.1f KB/file'
          % (format(len(files), ','), total / 1e9, avg / 1024))
    print('Index: %s usable players' % format(len(usable), ','))
    # Check mean output size to detect empty compressed files despite normal logs.
    # File counts alone cannot detect empty payloads (about 46 bytes after gzip).
    if avg < 2 * 1024:
        raise SystemExit('Average size only %.0f bytes; output appears empty' % avg)

In [ ]:
# One-time extraction. Later training runs can start at the next section.
t0 = time.time()
extract_all(df)
print('Completed; total time %.1f min' % ((time.time() - t0) / 60))

# 2. Data Loader

If extraction is complete, training can resume here after imports and configuration.

A training sample is a triplet:
```text
anchor     A window from one player
positive   A window from a different game of the same player
negative   A window from a different player
```

The network maps each window to a 256-dimensional embedding. Triplet loss is:
```text
max(0, d(anchor, positive) - d(anchor, negative) + margin)
```
The configured margin is 0.5. Triplets already satisfying the margin contribute zero loss.

Sampling chooses two players, two different positive-player games, and one window per selected game. Worker- and epoch-dependent seeds reduce repetition of the same random sampling sequence. They do not guarantee that individual triplets never recur by chance.


In [ ]:
def load_window(key, rng):
    """Load all windows from a game and return one random window."""
    with gzip.open('%s/%s.npy.gz' % (FEAT_DIR, key), 'rb') as f:
        arr = np.load(f)
    return arr[rng.integers(len(arr))]


class TripletDataset(IterableDataset):
    def __init__(self, index, steps_per_epoch, batch_size):
        self.players = list(index.keys())
        self.index = index
        self.samples = steps_per_epoch * batch_size
        self.epoch = 0

    def __iter__(self):
        info = torch.utils.data.get_worker_info()
        wid = info.id if info else 0
        nworkers = info.num_workers if info else 1

        # Vary the seed by both worker and epoch
        rng = np.random.default_rng(SEED + 1000 * self.epoch + wid)
        self.epoch += 1

        for _ in range(self.samples // nworkers):
            pos_player, neg_player = rng.choice(self.players, 2, replace=False)
            pos_games = self.index[pos_player]
            neg_games = self.index[neg_player]
            a_key, p_key = rng.choice(pos_games, 2, replace=False)
            n_key = neg_games[rng.integers(len(neg_games))]
            try:
                yield (load_window(a_key, rng),
                       load_window(p_key, rng),
                       load_window(n_key, rng))
            except (OSError, EOFError, ValueError):
                continue          # Skip missing or unreadable records

In [ ]:
with open(INDEX_PATH) as f:
    index = json.load(f)
print('Index: %s players / %s games'
      % (format(len(index), ','), format(sum(len(v) for v in index.values()), ',')))

train_dataset = TripletDataset(index, STEPS_PER_EPOCH, BATCH_SIZE)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, num_workers=NPROC,
                          pin_memory=True, persistent_workers=True, drop_last=True)

_a, _p, _n = next(iter(train_loader))
print('One batch:', _a.shape, _p.shape, _n.shape)

# 3. Model Training

`GoPlayerResNet` produces 256-dimensional L2-normalized embeddings with unit length. Normalization controls embedding scale so distances represent differences in direction and the margin remains interpretable.

Anchor, positive and negative tensors are concatenated for a single forward pass, then split for loss calculation. The baseline uses BF16 autocast on CUDA; the original tutorial reports that this combined approach was faster than three separate forward passes.


In [ ]:
model = GoPlayerResNet().to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
criterion = torch.nn.TripletMarginLoss(margin=MARGIN, p=2, reduction='mean')

print('Parameter count: {:,}'.format(sum(p.numel() for p in model.parameters())))
print('batch %d | lr %s | margin %s | %d epochs × %s steps'
      % (BATCH_SIZE, LEARNING_RATE, MARGIN, NUM_EPOCHS,
         format(STEPS_PER_EPOCH, ',')))

In [ ]:
def train_step(model, optimizer, criterion, batch):
    """One triplet step with a combined forward pass and BF16 autocast."""
    anchor, positive, negative = batch
    n = anchor.size(0)
    # Transfer uint8 to GPU before casting, reducing transfer size versus float32
    x = torch.cat([anchor, positive, negative], 0).to(DEVICE, non_blocking=True)
    with torch.autocast('cuda', dtype=torch.bfloat16, enabled=DEVICE.type == 'cuda'):
        a, p, neg = model(x.float()).split(n)
        loss = criterion(a, p, neg)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    return loss.item()

In [ ]:
# Keep the kernel running during training.
history = []
model.train()
best = float('inf')

for epoch in range(NUM_EPOCHS):
    t0 = time.time()
    run_loss, steps = 0.0, 0

    for batch in train_loader:
        run_loss += train_step(model, optimizer, criterion, batch)
        steps += 1
        if steps % LOG_INTERVAL == 0:
            print('  ep%d step %s  loss %.4f  %.1f min'
                  % (epoch + 1, format(steps, ','), run_loss / steps,
                     (time.time() - t0) / 60), flush=True)
        if steps >= STEPS_PER_EPOCH:
            break

    avg = run_loss / max(steps, 1)
    history.append({'epoch': epoch + 1, 'loss': avg,
                    'minutes': (time.time() - t0) / 60})
    print('Epoch %d/%d  loss %.4f  %.1f min'
          % (epoch + 1, NUM_EPOCHS, avg, (time.time() - t0) / 60), flush=True)

    if avg < best:
        best = avg
        torch.save(model.state_dict(), f'{CKPT_DIR}/best_model.pth')
        print('  New lowest loss %.4f; saved best_model.pth' % best)
    torch.save(model.state_dict(), f'{CKPT_DIR}/epoch_%d.pth' % (epoch + 1))

pd.DataFrame(history).to_csv(f'{CKPT_DIR}/history.csv', index=False)
print('Training completed; lowest loss %.4f' % best)

### Checking whether training is progressing

An untrained model often has triplet loss near the configured margin of 0.5 because positive and negative distances are initially similar.

A decreasing loss is a useful diagnostic. A flat loss near 0.5 warrants investigation of sampling, gradients and optimization; a different seed or lower learning rate can be tested. Loss alone does not establish identification quality.

The following cell plots the training history when available.


In [ ]:
import matplotlib.pyplot as plt

hist = pd.read_csv(f'{CKPT_DIR}/history.csv')

plt.figure(figsize=(8, 4))
plt.plot(hist['epoch'], hist['loss'], marker='o')
plt.axhline(MARGIN, ls='--', c='gray', lw=1)
plt.annotate('margin = %.1f (untrained level)' % MARGIN,
             xy=(hist['epoch'].iloc[0], MARGIN), xytext=(1.2, MARGIN - 0.02),
             color='gray')
plt.xlabel('epoch'); plt.ylabel('triplet loss'); plt.xticks(list(hist['epoch']))
plt.ylim(0, 0.55); plt.grid(alpha=.3); plt.tight_layout(); plt.show()


# 4. Inference with Testing Data On-the-Fly

**Candidate representation:** compute embeddings for the windows from each candidate player's approximately 100 games, then average all windows to form one centroid per player.

**Query matching:** average the query windows, calculate L2 distance to the 400 candidate centroids and return the five nearest players. Averaging many windows reduces variability from individual moves.

**Public and private questions must both be predicted and combined into one 800-row submission.** Candidate pools are separate: public questions require `pub_p_` players and private questions require `priv_p_` players.


In [ ]:
GPU_BATCH = 4096
CHUNK = 256
_infer_parser = SGFParsePlayerIdentification()


def sgf_windows(args):
    """Top-level multiprocessing.Pool helper: (sgf, color) -> (N, 17, 19, 19)"""
    sgf, color = args
    try:
        feats = _infer_parser.extract_features(sgf, str(color).lower(),
                                               mode='inference')
        return np.asarray(feats, dtype=np.uint8) if len(feats) else None
    except Exception:
        return None


def embed(model, arr):
    outs = []
    with torch.no_grad():
        for i in range(0, len(arr), GPU_BATCH):
            x = torch.from_numpy(arr[i:i + GPU_BATCH]).float().to(DEVICE)
            outs.append(model(x).cpu().numpy())
    return np.concatenate(outs) if outs else np.zeros((0, 256), np.float32)


def mean_embeddings(model, pool, jobs, keys, label):
    """Jobs and keys have equal length; average all windows for each key."""
    sums, counts = {}, {}
    t0 = time.time()
    for start in range(0, len(jobs), CHUNK):
        for key, arr in zip(keys[start:start + CHUNK],
                            pool.map(sgf_windows, jobs[start:start + CHUNK])):
            if arr is None:
                continue
            e = embed(model, arr)
            sums[key] = sums.get(key, 0) + e.sum(0)
            counts[key] = counts.get(key, 0) + len(e)
        done = min(start + CHUNK, len(jobs))
        if done % (CHUNK * 20) == 0 or done == len(jobs):
            el = time.time() - t0
            print('  %s %d/%d games, %.0f sec' % (label, done, len(jobs), el), flush=True)
    return {k: sums[k] / counts[k] for k in sums}


def predict(weights_path, out_path):
    net = GoPlayerResNet()
    net.load_state_dict(torch.load(weights_path, map_location='cpu'))
    net.eval().to(DEVICE)

    rows = []
    with Pool(NPROC) as pool:
        # Keep public and private candidate pools separate
        for split in ('public', 'private'):
            print('=== %s ===' % split, flush=True)
            cand = pd.read_csv(
                f'{TEST_DIR}/player_identification_candidates_{split}.csv')
            centroids = mean_embeddings(
                net, pool, list(zip(cand['sgf_content'], cand['color'])),
                list(cand['player_id']), 'Candidates')
            names = sorted(centroids)
            C = np.stack([centroids[n] for n in names])
            print('  Centroids: %d' % len(names), flush=True)

            test = pd.read_csv(f'{TEST_DIR}/player_identification_test_{split}.csv')
            sgf_cols = [c for c in test.columns if c.startswith('sgf_')]
            col_cols = [c for c in test.columns if c.startswith('color_')]

            jobs, keys = [], []
            for _, r in test.iterrows():
                # Filter empty sgf_ fields beyond num_games
                for sc, cc in zip(sgf_cols, col_cols):
                    if pd.notna(r[sc]) and str(r[sc]).strip():
                        jobs.append((r[sc], r[cc]))
                        keys.append(r['question_id'])
            queries = mean_embeddings(net, pool, jobs, keys, 'Queries')

            for qid in test['question_id']:
                q = queries.get(qid)
                if q is None:
                    rows.append([qid] + names[:TOP_N])
                    continue
                d = np.linalg.norm(C - q, axis=1)
                rows.append([qid] + [names[i] for i in np.argsort(d)[:TOP_N]])
            print('  %s completed: %d questions' % (split, len(test)), flush=True)

    sub = pd.DataFrame(rows,
                       columns=['question_id'] + ['top_%d' % i
                                                  for i in range(1, TOP_N + 1)])
    sub.to_csv(out_path, index=False)
    print('Wrote %s (%d rows)' % (out_path, len(sub)))
    return sub

In [ ]:
submission = predict(f'{CKPT_DIR}/best_model.pth', './submission_player.csv')
submission.head()

# End of the tutorial

The original reference run reported a public score of **0.2845**. This is historical reference evidence, not a result produced by this translated copy. Private scores are published after the competition.

| Component | Reference run |
|---|---|
| GPU | NVIDIA GeForce RTX 4090, 24 GB, driver 535.288.01 |
| CPU | AMD EPYC-Genoa, 8 cores |
| RAM | 31 GB |
| OS | Ubuntu 22.04.5 LTS |
| Software | Python 3.10.12, PyTorch 2.6.0+cu124, CUDA 12.4 |

This tutorial is a reference implementation. Architecture, features and training methods can be adapted according to the competition rules.

